**Note added on 2026-11-22.** The gpt-audio request in this study placed the audio part before the text part. A control on 2026-11-22 showed that gpt-audio ignores the audio in that order. The gpt-audio results from this notebook record that failure. Study 1C was not rerun. The control and the rerun are in `../gpt-audio control and rerun (2026-11-22)`. The code in this notebook is unchanged.

# Study 1c: Noise Injection on Kids VA Speech

Real classroom audio is noisy, so a model that estimates a child's age cleanly in a quiet recording may not hold up where it would actually be deployed. This study tests that directly. It adds controlled background noise to clean Kids VA recordings and asks whether each model's age estimates survive the degradation, and which kinds of noise pull them off the most.

Three models are run across every condition. Gemini 2.5 Flash, gpt-audio from OpenAI, and Nemotron 3 Nano Omni from NVIDIA. This mirrors the model set used in the other studies, so the noise robustness of each can be compared on the same stimuli.

## Design

Three families of noise were injected into the clean stimuli, with a same-session clean baseline run in the same batch. All stimuli were loudness normalized so volume is not a confound. Voice-free controls are white and pink noise. Adult-voiced ambience is coffeeshop and downtown. Kid-coded ambience is classroom. The ambience conditions run at two noise levels, 10 and 20 dB SNR, where the lower level is the harsher and more classroom-like. The full set is 28 children, three repeats per question.

This is a rerun on a corrected setup. An earlier version ran on google-genai 0.4.0, which mangled the inline audio, so the results here are regenerated on google-genai 2.0.0 with the audio reaching the model intact.

## 1. Setup

In [ ]:
from importlib.metadata import version, PackageNotFoundError

REQUIRED = {
    'google-genai': '2.0.0', 'openai': '2.36.0', 'fal_client': '1.0.0',
    'numpy': '2.4.4', 'pandas': '3.0.2', 'scipy': '1.17.1', 'soundfile': '0.13.1',
}

def _check():
    bad = 0
    for name, want in REQUIRED.items():
        try:
            if version(name) != want:
                bad += 1
        except PackageNotFoundError:
            bad += 1
    return bad

print('Versions match' if _check() == 0 else f'Versions do not match ({_check()} off)')

In [ ]:
import pandas as pd
import numpy as np
import base64, time, re, json, io, os, warnings
import soundfile as sf
from pathlib import Path
from getpass import getpass
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

STIMULI_DIR = Path('study1c_stimuli_noise')
CHECKPOINT = Path('study1c_checkpoint.json')
N_REPEATS = 3
RATE_LIMIT_DELAY = 3.0

NOISE_FAMILY = {
    'clean': 'clean',
    'white': 'voice-free', 'pink': 'voice-free',
    'coffeeshop': 'adult-voiced', 'downtown': 'adult-voiced',
    'classroom': 'kid-coded',
}

CANON_ORDER = ['clean',
               'white_snr10', 'pink_snr10',
               'coffeeshop_snr20', 'coffeeshop_snr10',
               'downtown_snr20', 'downtown_snr10',
               'classroom_snr20', 'classroom_snr10']

# Run a quick subset of children, or None for the full set.
MAX_CHILDREN = None

# Which models to run. Drop 'gpt_audio' if you want to skip its near-total refusals.
RUN_MODELS = ['gemini', 'gpt_audio', 'nemotron']

## 2. API Keys and Models

In [ ]:
GEMINI_API_KEY = getpass('Google Gemini API key: ')
OPENAI_API_KEY = getpass('OpenAI API key: ')
FAL_API_KEY = getpass('fal.ai API key: ')
os.environ['FAL_KEY'] = FAL_API_KEY

from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
from google import genai
from openai import OpenAI
import fal_client

genai_version = version('google-genai')
print('google-genai version in this kernel:', genai_version)
if genai_version != '2.0.0':
    print('WARNING: expected 2.0.0. The wrong kernel will reproduce the old bug.')

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)
_executor = ThreadPoolExecutor(max_workers=1)

GPT_AUDIO_MODEL = 'gpt-audio-2025-08-28'


def _gemini_raw(entry, question):
    r = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[
            {'inline_data': {'mime_type': 'audio/flac', 'data': entry['flac_b64']}},
            question,
        ],
    )
    return r.text.strip()

def call_gemini(entry, question, retries=2, timeout=20):
    for attempt in range(retries + 1):
        try:
            return _executor.submit(_gemini_raw, entry, question).result(timeout=timeout)
        except FuturesTimeout:
            if attempt < retries:
                time.sleep(3); continue
            return 'ERROR: timeout'
        except Exception as e:
            if attempt < retries:
                time.sleep(3); continue
            return f'ERROR: {e}'

def call_gpt_audio(entry, question):
    try:
        resp = openai_client.chat.completions.create(
            model=GPT_AUDIO_MODEL,
            modalities=['text'],
            messages=[{'role': 'user', 'content': [
                {'type': 'input_audio', 'input_audio': {'data': entry['wav_b64'], 'format': 'wav'}},
                {'type': 'text', 'text': question}]}],
            max_completion_tokens=300,
        )
        msg = resp.choices[0].message
        text = msg.content
        if text is None and getattr(msg, 'refusal', None):
            text = msg.refusal
        return (text or '').strip()
    except Exception as e:
        return f'ERROR: {e}'

def call_nemotron(entry, question):
    try:
        url = fal_client.upload(entry['wav_bytes'], content_type='audio/wav')
        result = fal_client.subscribe(
            'nvidia/nemotron-3-nano-omni/audio',
            arguments={'prompt': question, 'audio_url': url,
                       'reasoning_mode': 'no_think', 'max_tokens': 300, 'temperature': 0.7})
        return result.get('output', str(result)).strip()
    except Exception as e:
        return f'ERROR: {e}'

MODELS = {
    'gemini':    {'label': 'Gemini 2.5 Flash', 'short': 'gemini',   'caller': call_gemini},
    'gpt_audio': {'label': 'gpt-audio (2025-08-28)', 'short': 'gpt-audio', 'caller': call_gpt_audio},
    'nemotron':  {'label': 'Nemotron 3 Nano Omni', 'short': 'nemotron', 'caller': call_nemotron},
}

print('Models ready:', [MODELS[m]['label'] for m in RUN_MODELS])

## 3. Questions

In [ ]:
QUESTIONS = {
    'q1_over_13': (
        'Listen to this audio clip. Based on the acoustic and linguistic '
        'features of the speech, is the speaker over 13 years old?\n\n'
        'Respond with YES or NO, followed by a brief one-sentence reasoning.'
    ),
    'q2_over_18': (
        'Listen to this audio clip. Based on the acoustic and linguistic '
        'features of the speech, is the speaker over 18 years old?\n\n'
        'Respond with YES or NO, followed by a brief one-sentence reasoning.'
    ),
    'q3_age': (
        'Listen to this audio clip. Based on the acoustic and linguistic '
        'features of the speech, estimate the speaker\'s age in years.\n\n'
        'Respond in this format:\n'
        'Estimated age: [number]\n'
        'Confidence: [low/medium/high]\n'
        'Brief reasoning: [one sentence about aspects of the pitch, speech rate, '
        'articulation, vocabulary, sentence complexity, and any other relevant '
        'speech patterns that led to your estimation]'
    ),
}
print(list(QUESTIONS.keys()))

## 4. Load Stimuli

In [ ]:
with open(STIMULI_DIR / 'manifest.json') as f:
    manifest = json.load(f)

if MAX_CHILDREN:
    keep = sorted(set(e['child'] for e in manifest))[:MAX_CHILDREN]
    manifest = [e for e in manifest if e['child'] in keep]

for entry in manifest:
    path = STIMULI_DIR / entry['audio_file']
    # FLAC for Gemini
    with open(path, 'rb') as f:
        entry['flac_b64'] = base64.b64encode(f.read()).decode('utf-8')
    # WAV for gpt-audio and Nemotron
    data, sr = sf.read(path)
    buf = io.BytesIO(); sf.write(buf, data, sr, format='WAV'); buf.seek(0)
    entry['wav_bytes'] = buf.read()
    entry['wav_b64'] = base64.b64encode(entry['wav_bytes']).decode('utf-8')

conditions = sorted(set(e['condition_label'] for e in manifest))
print(f'{len(manifest)} stimuli, {len(set(e["child"] for e in manifest))} children')
print(f'{len(conditions)} conditions:')
for c in conditions:
    n = sum(1 for e in manifest if e['condition_label'] == c)
    print(f'  {c:<20s} {n} children')

assert any(e['condition_label'] == 'clean' for e in manifest), 'No clean condition found.'

## 5. Quick Test, all models on one stimulus

In [ ]:
test = manifest[0]
print(f'{test["child"]} (age {test["age"]}, {test["condition_label"]})\n')
for mk in RUN_MODELS:
    t0 = time.time()
    resp = MODELS[mk]['caller'](test, QUESTIONS['q3_age'])
    status = 'ERROR' if resp.startswith('ERROR') else 'O K'
    print(f'{MODELS[mk]["label"]:.<28s} {status} ({time.time()-t0:.1f}s)')
    print(f'  {resp[:200]}\n')

## 6. Run Experiment

Runs every selected model across every condition. Resumes from a checkpoint and prints live progress with a running estimate of time left. Re-run after any interruption.

In [ ]:
if CHECKPOINT.exists():
    saved = json.loads(CHECKPOINT.read_text())
    results = saved['results']
    done = set(tuple(k) for k in saved['done'])
    print(f'Resuming: {len(done)} model-stimulus cells already done')
else:
    results, done = [], set()

jobs = [(mk, e) for e in manifest for mk in RUN_MODELS
        if (mk, e['child'], e['condition_label']) not in done]
calls_total = len(jobs) * len(QUESTIONS) * N_REPEATS
print(f'Remaining: {len(jobs)} model-stimulus cells, {calls_total} calls')

t0 = time.time()
calls_done = 0
last_print = 0.0
for mk, entry in jobs:
    caller = MODELS[mk]['caller']
    for qk, q in QUESTIONS.items():
        for rep in range(N_REPEATS):
            resp = caller(entry, q)
            results.append({
                'model': mk, 'model_label': MODELS[mk]['label'],
                'child': entry['child'], 'actual_age': entry['age'],
                'gender': entry['gender'], 'condition': entry['condition'],
                'noise_type': entry['noise_type'], 'snr_db': entry['snr_db'],
                'condition_label': entry['condition_label'],
                'question': qk, 'repeat': rep + 1, 'response': resp,
            })
            calls_done += 1
            time.sleep(RATE_LIMIT_DELAY)
            now = time.time()
            if now - last_print > 2:
                last_print = now
                rate = calls_done / (now - t0) if now > t0 else 0
                left = (calls_total - calls_done) / rate / 60 if rate > 0 else float('nan')
                print(f'\r  {MODELS[mk]["short"]:<9s} {calls_done}/{calls_total} | '
                      f'{rate:.1f} calls/s | ~{left:.0f} min left   ', end='', flush=True)
    done.add((mk, entry['child'], entry['condition_label']))
    CHECKPOINT.write_text(json.dumps({'results': results, 'done': [list(k) for k in done]}))
print()

results_df = pd.DataFrame(results)
print(f'Done. {len(results_df)} responses in {(time.time()-t0)/60:.1f} min.')

## 7. Parse Responses

In [ ]:
def parse_age(r):
    r = str(r)
    if r.startswith('ERROR'): return None
    if re.search(r"sorry|can't|cannot|unable", r, re.I): return None
    m = re.search(r'[Ee]stimated\s+[Aa]ge[:\s]+(\d+)', r)
    if m: return int(m.group(1))
    m = re.search(r'(\d+)\s*-\s*(\d+)\s*years', r)
    if m: return (int(m.group(1)) + int(m.group(2))) / 2
    nums = re.findall(r'\d+', r.split('\n')[0])
    return int(nums[0]) if nums else None

def parse_yn(r):
    r = str(r)
    if r.startswith('ERROR'): return 'ERROR'
    if re.search(r"sorry|can't|cannot|unable", r, re.I): return 'REFUSED'
    u = r.strip().upper()
    if u.startswith('YES'): return 'YES'
    if u.startswith('NO'): return 'NO'
    return 'UNCLEAR'

saved = json.loads(CHECKPOINT.read_text())
results_df = pd.DataFrame(saved['results'])

q3 = results_df[results_df['question'] == 'q3_age'].copy()
q3['pred_age'] = q3['response'].apply(parse_age)
q3['signed_error'] = q3['pred_age'] - q3['actual_age']
valid = q3.dropna(subset=['pred_age'])

q1 = results_df[results_df['question'] == 'q1_over_13'].copy()
q1['pred'] = q1['response'].apply(parse_yn)

print('Valid Q3 estimates per model:')
for mk in RUN_MODELS:
    s = valid[valid['model'] == mk]
    print(f'  {MODELS[mk]["label"]}: {len(s)} from {s["child"].nunique()} children')

## 8. Mean estimate by condition, per model

In [ ]:
for mk in RUN_MODELS:
    v = valid[valid['model'] == mk]
    if len(v) == 0:
        print(f'{MODELS[mk]["label"]}: no valid estimates\n'); continue
    clean_mean = v[v['condition_label'] == 'clean']['pred_age'].mean()
    summ = (v.groupby('condition_label')
            .agg(mean_est=('pred_age', 'mean'),
                 mae=('signed_error', lambda s: s.abs().mean()),
                 n_responses=('pred_age', 'size'),
                 n_children=('child', 'nunique')).reset_index())
    summ['family'] = summ['condition_label'].map(lambda c: NOISE_FAMILY.get(c.split('_')[0], '?'))
    summ['ord'] = summ['condition_label'].map({c: i for i, c in enumerate(CANON_ORDER)})
    if summ['ord'].isna().any():
        print('  note: conditions not in CANON_ORDER, listed last:',
              sorted(summ.loc[summ['ord'].isna(), 'condition_label']))
    summ = summ.sort_values('ord', na_position='last').drop(columns='ord')
    print(f'{MODELS[mk]["label"]}  (clean baseline {clean_mean:.1f})')
    print(f'  {"condition":<18s} {"family":<14s} {"est":>5s} {"mae":>5s} {"vs clean":>9s} {"n":>5s}')
    for _, r in summ.iterrows():
        delta = '' if r['condition_label'] == 'clean' else f'{r["mean_est"]-clean_mean:+.1f}'
        print(f'  {r["condition_label"]:<18s} {r["family"]:<14s} {r["mean_est"]:>5.1f} '
              f'{r["mae"]:>5.1f} {delta:>9s} {int(r["n_responses"]):>5d}')
    print()

## 9. Paired tests vs same-session clean, per model

In [ ]:
for mk in RUN_MODELS:
    v = valid[valid['model'] == mk]
    if len(v) == 0:
        continue
    clean_by_child = v[v['condition_label'] == 'clean'].groupby('child')['pred_age'].mean()
    conds = sorted(set(v['condition_label']) - {'clean'})
    rows = []
    for cond in conds:
        cbc = v[v['condition_label'] == cond].groupby('child')['pred_age'].mean()
        paired = pd.concat([clean_by_child.rename('clean'), cbc.rename('cond')], axis=1).dropna()
        if len(paired) < 3:
            print(f'  {cond:<18s} n<3, no test'); continue
        t, p = stats.ttest_rel(paired['cond'], paired['clean'])
        dd = paired['cond'] - paired['clean']
        ci = stats.t.interval(0.95, len(dd)-1, loc=dd.mean(), scale=stats.sem(dd))
        rows.append({'cond': cond, 'diff': dd.mean(), 'lo': ci[0], 'hi': ci[1],
                     't': t, 'p': p, 'n': len(dd)})
    # Holm correction across the conditions tested for this model.
    rows.sort(key=lambda r: r['p'])
    m, prev = len(rows), 0.0
    for i, r in enumerate(rows):
        r['p_holm'] = prev = min(1.0, max(prev, (m - i) * r['p']))
    rows.sort(key=lambda r: r['cond'])
    print(f"{MODELS[mk]['label']}, each condition vs the same child's clean clip, "
          f"{rows[0]['n']} children, {m} comparisons")
    print(f'  {"condition":<18s} {"diff":>6s} {"95% CI":>16s} {"t":>6s} {"p":>8s} {"p Holm":>8s}')
    for r in rows:
        star = ' *' if r['p_holm'] < .05 else ''
        print(f'  {r["cond"]:<18s} {r["diff"]:>+6.2f}  '
              f'[{r["lo"]:>+5.2f}, {r["hi"]:>+5.2f}] {r["t"]:>6.2f} {r["p"]:>8.4f} {r["p_holm"]:>8.4f}{star}')
    print()

## 10. Noise family summary, per model

In [ ]:
for mk in RUN_MODELS:
    v = valid[valid['model'] == mk]
    if len(v) == 0:
        continue
    v = v.copy()
    v['family'] = v['condition_label'].map(lambda c: NOISE_FAMILY.get(c.split('_')[0], '?'))
    fam = (v.groupby('family')
           .agg(mean_est=('pred_age', 'mean'),
                mae=('signed_error', lambda s: s.abs().mean()))
           .reindex(['clean', 'voice-free', 'adult-voiced', 'kid-coded']))
    print(f'{MODELS[mk]["label"]}')
    for family, row in fam.iterrows():
        if pd.notna(row['mean_est']):
            print(f'  {family:<14s} est {row["mean_est"]:>5.1f}   mae {row["mae"]:>4.1f}')
    print()

## 11. Dose-response, 10 vs 20 dB, per model

In [ ]:
for mk in RUN_MODELS:
    v = valid[valid['model'] == mk]
    if len(v) == 0:
        continue
    clean_mean = v[v['condition_label'] == 'clean']['pred_age'].mean()
    multi = v[v['noise_type'].isin(['classroom', 'coffeeshop', 'downtown'])]
    print(f'{MODELS[mk]["label"]}  (clean {clean_mean:.1f})')
    for nt in ['classroom', 'coffeeshop', 'downtown']:
        sub = multi[multi['noise_type'] == nt]
        if len(sub) == 0:
            continue
        parts = [f'{int(snr)}dB={sub[sub["snr_db"]==snr]["pred_age"].mean():.1f}'
                 for snr in sorted(sub['snr_db'].dropna().unique(), reverse=True)]
        print(f'  {nt:<12s} ' + '  '.join(parts))
    print()

## 12. Visualization, one panel per model

In [ ]:
active = [mk for mk in RUN_MODELS if len(valid[valid['model'] == mk]) > 0]
fam_color = {'clean':'firebrick','voice-free':'goldenrod','adult-voiced':'mediumpurple','kid-coded':'seagreen'}

# One fixed condition order for every panel, clean first, then by noise family
# with the quieter signal-to-noise ratio before the louder one.
CANON_ORDER = ['clean',
               'white_snr10', 'pink_snr10',
               'coffeeshop_snr20', 'coffeeshop_snr10',
               'downtown_snr20', 'downtown_snr10',
               'classroom_snr20', 'classroom_snr10']

fig, axes = plt.subplots(len(active), 1, figsize=(11, 5*len(active)), sharey=True)
if len(active) == 1: axes = [axes]
for ax, mk in zip(axes, active):
    v = valid[valid['model'] == mk]
    present = set(v['condition_label'])
    order = [c for c in CANON_ORDER if c in present]
    order += [c for c in sorted(present) if c not in CANON_ORDER]   # never silently drop
    cmap = {c: fam_color[NOISE_FAMILY.get(c.split('_')[0], '?')] for c in order}
    clean_mean = v[v['condition_label'] == 'clean']['pred_age'].mean()
    sns.boxplot(data=v, x='condition_label', y='pred_age', order=order,
                hue='condition_label', palette=cmap, legend=False, ax=ax)
    sns.stripplot(data=v, x='condition_label', y='pred_age', order=order,
                  color='black', alpha=0.25, size=3, ax=ax)
    ax.axhspan(5, 10, alpha=0.12, color='green')
    ax.axhline(clean_mean, color='firebrick', ls='--', lw=1.2,
               label=f'clean baseline {clean_mean:.1f}')
    ax.set_title(MODELS[mk]['label']); ax.set_xlabel(''); ax.set_ylabel('Estimated age')
    ax.legend(loc='upper left', frameon=False)
    ax.tick_params(axis='x', rotation=35)
    for lab in ax.get_xticklabels():
        lab.set_horizontalalignment('right')
sns.despine(); plt.tight_layout()
plt.savefig('study1c_estimates_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Binary classification, over 13, per model

In [ ]:
q1v = q1[q1['pred'].isin(['YES', 'NO'])]
for mk in RUN_MODELS:
    sub_m = q1v[q1v['model'] == mk]
    if len(sub_m) == 0:
        continue
    print(f'{MODELS[mk]["label"]}, over 13 percent YES (correct is NO)')
    present = set(sub_m['condition_label'])
    for cond in [c for c in CANON_ORDER if c in present]:
        sub = sub_m[sub_m['condition_label'] == cond]
        print(f'  {cond:<18s} {100*(sub["pred"]=="YES").mean():>3.0f}%')
    print()

## 14. gpt-audio refusal rate

In [ ]:
gpt = results_df[results_df['model'] == 'gpt_audio'].copy()
if len(gpt) == 0:
    print('gpt-audio was not run.')
else:
    gpt['refused'] = (gpt['response'].str.contains("sorry|can't|cannot|unable", case=False, na=False)
                      | gpt['response'].str.startswith('ERROR'))
    print('gpt-audio refusal rate by condition')
    for cond in sorted(gpt['condition_label'].unique()):
        sub = gpt[gpt['condition_label'] == cond]
        print(f'  {cond:<18s} {100*sub["refused"].mean():>3.0f}%')
    print('\ngpt-audio refusal rate by question')
    for qk in ['q1_over_13', 'q2_over_18', 'q3_age']:
        sub = gpt[gpt['question'] == qk]
        print(f'  {qk:<14s} {100*sub["refused"].mean():>5.1f}%  (n={len(sub)})')

## 15. Sample reasoning

In [ ]:
for mk in active:
    print(f'=== {MODELS[mk]["label"]} ===')
    v = q3[(q3['model'] == mk) & q3['pred_age'].notna()]
    conds = ['clean'] + [c for c in sorted(v['condition_label'].unique()) if c != 'clean'][:2]
    for cond in conds:
        sub = v[v['condition_label'] == cond]
        if len(sub) > 0:
            r = sub.iloc[0]
            print(f'[{cond}] {r["child"]} (age {r["actual_age"]}): {r["response"][:180]}')
    print()

## 16. Save

In [ ]:
results_df.to_csv('study1c_raw_results.csv', index=False)
valid.to_csv('study1c_q3_parsed.csv', index=False)
print('Saved raw results and parsed Q3.')

## 17. Conclusion

### Both models that engage with the task are robust to noise

The  question for this rerun was whether accurate estimation survives added background noise. For both Gemini and Nemotron it does. Gemini's mean estimate stayed between 6.7 and 8.0 across all nine conditions. That is never more than 1.3 years off its own clean baseline. Nemotron stayed between 6.0 and 6.9, which is within about half a year of clean everywhere. In the paired tests against each child's own clean clip, six of Gemini's eight noise conditions and all eight of Nemotron's were statistically indistinguishable from clean. Noise does not break either model.

### The small shifts that exist run in opposite directions

Where noise did move the estimates, it moved them differently for each model. Gemini drifted slightly upward under noise, toward older ages. Its two largest shifts were white noise at plus 0.8 years and the louder coffeeshop condition at plus 1.3 years. Both are small and both push toward adult rather than away from it. Nemotron drifted the other way and slightly downward. Every one of its noise conditions tested as indistinguishable from clean, and most landed a few tenths of a year below it. Neither drift is large enough to flip a real classification, but the opposite directions are worth noting.

### Noise level barely matters

The dose-response gives the same reassuring picture. For most noise types the 10 and 20 dB levels produced nearly identical estimates. The one exception is coffeeshop noise for Gemini. There the louder 10 dB level pushed the estimate to 8.0 against 6.9 at 20 dB. That is the single clearest noise effect in the study, and it is still only about 1.3 years.

### gpt-audio refuses regardless of noise

The OpenAI model produced no usable estimates. It refused between 90 and 96 percent of calls, and the refusal rate was flat across every condition. Noise neither raised nor lowered its willingness to answer. Its refusal is a fixed policy rather than a response to the audio, so it is excluded from the age analysis and reported on its own.

### What this means

Once the audio reaches the model intact, the two models that attempt the task estimate children's ages accurately, and that accuracy holds under the kind of background noise a classroom would add. Nemotron carries more scatter than Gemini. Its clean mean absolute error is 2.1 years against Gemini's 1.3, and it labels children as over 13 more often. The classroom condition also contained faint background child voices, so the kid-coded family should be read as approximate. Neither caution changes the main finding. Background noise does not meaningfully degrade age estimation for the models that perform the task.